In [2]:
"""
GPU DM-Soliton Simulator | v4.7-Final (JCP Submission Ready)
================================================================================
PAPER TITLE: Eliminating Spurious First-Order Convergence in Split-Step Fourier
             Simulations of Dispersion-Managed Solitons

CHANGELOG v4.3-final -> v4.7-Final:
  [FIX 1] _exp64: alpha factor verified (v4.3 was CORRECT all along).
          D(L) = exp(-alpha*L/2 + i*beta2*w^2*L/2).  With L=dz/2, hd=dz/2:
            D(dz/2) = exp(-alpha*dz/4 + i*beta2*w^2*dz/4)
                    = exp(-0.5*alpha*hd + 0.5j*beta2*w^2*hd)  [CORRECT]
          Each half-step carries loss exp(-alpha*dz/4); two half-steps give
          total per-step amplitude decay exp(-alpha*dz/2), i.e. power decay
          exp(-alpha*dz).
  [FIX 2] Case 11 lossy diagnostic: E_expected = E0*exp(-alpha*z) (CORRECT).
          _energy() = integral |A|^2 dt.  Since |A| decays as exp(-alpha*z/2),
          |A|^2 decays as exp(-alpha*z), so E(z) = E0*exp(-alpha*z).
  [FIX 3] Case 12 REDESIGNED: K=11 eval-grid sweep replaced by K=10
          physical interface sweep with EXACT ANALYTIC reference (gamma=0).
          L1_eff = 40 + delta*dz moves the physical interface within step j=4.
          Traditional: eval at step start (j*dz=40) always assigns N-fiber
            → error ∝ (1-delta), R²=1.000
          Midpoint: eval at step centre ((j+0.5)*dz=45) assigns correctly
            for delta<0.5, wrongly for delta>0.5
            → error ∝ min(delta, 1-delta), R²=1.000
          Both methods show delta-dependent error with perfect analytic fits.
          Key insight: Traditional always worse than or equal to Midpoint.
  [FIX 4] _grid(log=True): now calls _log_minor_ticks so log-axis minor
          grid lines are drawn (was silent pass in v4.3).
  [FIX 5] export_all_results: Case10 and Case11 conv data now exported
          (both were missing from conv_map in original v4.3).
  [FIX 6] omega_g / t_gpu float32 dead code removed; w_64 used exclusively.
  [FIX 7] plot_panel_case1: separate zr_t / zr_m x-arrays for panels (b)(c)
          (avoids silent length mismatch when traditional/midpoint Nz differ).
  [IMPR 1] LogLocator(subs='all') shows all decade subdivisions.
  [IMPR 2] _fit_slope bounds guard: n = min(n, len(x)).
  [IMPR 3] Case 1 panel (b): midpoint uses its own zr2 x-axis.
  [IMPR 4] Case 12: exports .npz with full fit data for both methods.

ALL FIGURES MERGED INTO PANELS -- No empty subplots, 1x3 / 1x2 layouts.
COMPLETE VERSION: Sections 0-11, including Case 0-12 + .npz export.
"""

import os
import time
import numpy as np
import cupy as cp
from tqdm import tqdm
from scipy.stats import linregress
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.font_manager as fm
import warnings

# ==============================================================================
# 0. OUTPUT DIRECTORY
# ==============================================================================
FIG_DIR = "./figures_jcp"
os.makedirs(FIG_DIR, exist_ok=True)

def figpath(fname):
    base = os.path.splitext(fname)[0]
    return os.path.join(FIG_DIR, base + ".pdf")

# ==============================================================================
# 1. JOURNAL-QUALITY STYLE (JCP / AIP standard)
# ==============================================================================
def _configure_fonts():
    available = {f.name for f in fm.fontManager.ttflist}
    preferred = ["Times New Roman", "DejaVu Serif", "Liberation Serif",
                 "Nimbus Roman", "Computer Modern Roman", "Noto Serif"]
    found = [f for f in preferred if f in available]
    serif_fallback = found if found else ["serif"]
    plt.rcParams.update({
        "font.family":        serif_fallback,
        "font.size":          9,
        "axes.labelsize":     9,
        "xtick.labelsize":    8,
        "ytick.labelsize":    8,
        "legend.fontsize":    7.5,
        "axes.titlesize":     9,
        "lines.linewidth":    1.2,
        "lines.markersize":   5,
        "patch.linewidth":    0.8,
        "axes.linewidth":     0.8,
        "axes.spines.top":    True,
        "axes.spines.right":  True,
        "xtick.direction":    "in",
        "ytick.direction":    "in",
        "xtick.major.size":   4.0,  "xtick.minor.size":  2.0,
        "ytick.major.size":   4.0,  "ytick.minor.size":  2.0,
        "xtick.major.width":  0.8,  "xtick.minor.width": 0.6,
        "ytick.major.width":  0.8,  "ytick.minor.width": 0.6,
        "xtick.top":          True,
        "ytick.right":        True,
        "figure.dpi":         150,
        "savefig.dpi":        600,
        "savefig.bbox":       "tight",
        "savefig.pad_inches": 0.02,
        "pdf.fonttype":       42,
        "ps.fonttype":        42,
    })
    print(f"[Font] Serif family: {serif_fallback[0]}")
    if "Times New Roman" not in available:
        print("[Font] 'Times New Roman' not found; graceful fallback applied.")

_configure_fonts()
warnings.filterwarnings("ignore", category=UserWarning, module="matplotlib")

W1 = 3.375
W2 = 6.75
H1 = W1 / 1.618
H2 = W2 / 1.618

C_TRAD  = "#C0392B"
C_MID   = "#2166AC"
C_SS4   = "#1A9641"
C_REF1  = "#888888"
C_REF2  = "#333333"
C_REF4  = "#762A83"

def _log_minor_ticks(ax):
    # [FIX 3] properly set log minor ticks on log-scale axes
    for axis in [ax.xaxis, ax.yaxis]:
        if axis.get_scale() == 'log':
            axis.set_minor_locator(ticker.LogLocator(subs='all', numticks=12))
        else:
            axis.set_minor_locator(ticker.AutoMinorLocator())

def _minor_ticks_linear(ax):
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())

def _grid(ax, log=False):
    ax.grid(True, which="major", lw=0.4, color="0.85", zorder=0)
    if log:
        # [FIX 3] was silent pass; now draws minor grid on log axes
        _log_minor_ticks(ax)
        ax.grid(True, which="minor", lw=0.18, color="0.93", zorder=0)
    else:
        ax.grid(True, which="minor", lw=0.25, color="0.92", zorder=0)

def _savefig(fig, out):
    fig.tight_layout(pad=0.4)
    fig.savefig(out, format="pdf")
    print(f"[PLOT] -> {out}")
    plt.close(fig)

# ==============================================================================
# 2. PHYSICAL CONSTANTS & PARAMETERS
# ==============================================================================
c        = 299792458.0
lambda0  = 1550e-9
omega0   = 2 * np.pi * c / lambda0
n2       = 2.6e-20
A_eff    = 80e-12
gamma    = n2 * omega0 / (c * A_eff)
alpha_0  = 0.0
beta2_n  = +20e-27
beta2_a  = -20e-27
T0       = 10e-12
dz_base  = 0.5
T_window = 200 * T0
Nt       = 2**14
dt       = T_window / Nt
L_D      = T0**2 / abs(beta2_n)
P0_N1    = abs(beta2_a) / (gamma * T0**2)

sep   = "=" * 80
sep_s = "-" * 80

print(sep)
print("  DM SOLITON SSFM -- v4.7-Final REFEREE-PROOF VALIDATION SUITE")
print(sep)
try:
    gpu_name = cp.cuda.runtime.getDeviceProperties(0)["name"].decode()
    mem_free, mem_total = cp.cuda.Device(0).mem_info
    print(f"\n[GPU] {gpu_name} | {mem_total/1024**3:.1f} GB total | {mem_free/1024**3:.2f} GB free")
except Exception as e:
    print(f"\n[GPU] Query failed: {e}")
print(f"\n  gamma={gamma:.3e} 1/(W*m)  |  L_D={L_D:.0f} m  |  P0_N1={P0_N1:.4f} W")
print(f"  Figures -> {os.path.abspath(FIG_DIR)}/")
print(sep_s)

# ==============================================================================
# 3. GPU GRIDS  ([FIX 5] float32 dead code removed)
# ==============================================================================
t_64 = cp.linspace(-T_window/2, T_window/2, Nt, dtype=cp.float64)
w_64 = (2*np.pi*cp.fft.fftfreq(Nt, dt)).astype(cp.float64)

# ==============================================================================
# 4. SSFM OPERATORS
# ==============================================================================
def _exp64(beta2, dz, alpha=0.0):
    """
    Half-step dispersion + loss operator for symmetric split-step (D/2--NL--D/2).

    D(L) = exp(-alpha*L/2 + i*beta2*w^2*L/2).  With L = dz/2, hd = dz/2:
      dispersion: exp(i * beta2 * omega^2 * dz/4)   [0.5j*beta2*w^2*hd]
      loss:       exp(-alpha * dz/4)                 [0.5*alpha*hd]

    Two half-steps give total per-step amplitude decay exp(-alpha*dz/2),
    i.e. power decay exp(-alpha*dz).  This is the physically correct result.
    """
    hd = dz / 2.0
    return cp.exp((-0.5 * alpha * hd) + (0.5j * beta2 * w_64**2) * hd).astype(cp.complex128)

def _nl64(dz, gamma_val=None):
    g = gamma_val if gamma_val is not None else gamma
    return cp.complex128(1j * g * dz)

def _ssfm64(A, el, nl):
    A = cp.fft.ifft(cp.fft.fft(A) * el)
    A *= cp.exp(nl * (A * cp.conj(A)).real)
    A = cp.fft.ifft(cp.fft.fft(A) * el)
    return A

_Y_c1 = 1.0 / (2.0 - 2.0**(1.0/3.0))
_Y_c0 = 1.0 - 2.0*_Y_c1
_YOSHIDA_C = [_Y_c1, _Y_c0, _Y_c1]

def _ssfm64_ss4(A, beta2, dz, alpha=0.0):
    for c in _YOSHIDA_C:
        el = _exp64(beta2, c*dz, alpha)
        nl = _nl64(c*dz)
        A = _ssfm64(A, el, nl)
    return A

# ==============================================================================
# 5. PULSE FACTORIES
# ==============================================================================
def _pulse64(p0):
    return cp.sqrt(cp.float64(p0)) * cp.exp(-0.5*(t_64/T0)**2).astype(cp.complex128)

def _pulse64_sech(p0):
    return cp.sqrt(cp.float64(p0)) / cp.cosh(t_64/T0).astype(cp.complex128)

def _pulse64_two(p0, separation=6.0, phase_diff=np.pi):
    P0_f = float(p0)
    A1 = cp.sqrt(P0_f/2) * cp.exp(-0.5*((t_64 - separation*T0)/T0)**2)
    A2 = cp.sqrt(P0_f/2) * cp.exp(-0.5*((t_64 + separation*T0)/T0)**2) * cp.exp(1j*phase_diff)
    return (A1 + A2).astype(cp.complex128)

# ==============================================================================
# 6. INDEXING
# ==============================================================================
def build_index(Ltot, dz, L1, L2, method='traditional'):
    L_map  = L1 + L2
    Nz     = int(np.round(Ltot/dz)) + 1
    k      = np.arange(1, Nz)
    z_eval = k*dz if method == 'traditional' else (k-0.5)*dz
    idx    = np.zeros(Nz, dtype=np.int8)
    idx[1:] = (z_eval % L_map >= L1).astype(np.int8)
    return Nz, idx

def make_indexer(method):
    return lambda Ltot, dz, L1, L2: build_index(Ltot, dz, L1, L2, method)

# ==============================================================================
# 7. SIMULATION RUNNERS
# ==============================================================================
def _energy(A):
    return float(cp.sum(cp.abs(A)**2) * dt)

def _l2rel(A, R):
    return float(cp.sqrt(cp.sum(cp.abs(A - R)**2) / cp.sum(cp.abs(R)**2)))

def _peak(A):
    return float(cp.max(cp.abs(A)**2))

def _fwhm(A):
    """Returns FWHM / T0 (dimensionless)."""
    I   = cp.abs(A)**2
    Im  = cp.max(I)
    idx = cp.where(I > 0.5*Im)[0]
    return float((idx[-1] - idx[0]) * dt / T0) if len(idx) > 0 else 0.0

def run64(dz, indexer, Ltot, P0, L1, L2, pulse='gaussian', alpha=0.0, **kw):
    Nz, idx = indexer(Ltot, dz, L1, L2)
    en = _exp64(beta2_n, dz, alpha)
    ea = _exp64(beta2_a, dz, alpha)
    nl = _nl64(dz)
    A  = {'gaussian': _pulse64, 'sech': _pulse64_sech, 'two': _pulse64_two}[pulse](P0, **kw)
    for i in range(1, Nz):
        A = _ssfm64(A, ea if idx[i] else en, nl)
        if not cp.isfinite(cp.sum(A)):
            raise RuntimeError(f"NaN/Inf at step {i}, z={i*dz:.3f} m")
    return A

def run64_diag(dz, indexer, Ltot, P0, L1, L2,
               pulse='gaussian', alpha=0.0, rec=100, **kw):
    Nz, idx = indexer(Ltot, dz, L1, L2)
    en = _exp64(beta2_n, dz, alpha)
    ea = _exp64(beta2_a, dz, alpha)
    nl = _nl64(dz)
    A  = {'gaussian': _pulse64, 'sech': _pulse64_sech, 'two': _pulse64_two}[pulse](P0, **kw)
    E0 = _energy(A)
    z_r, e_r, p_r, f_r = [], [], [], []
    for i in range(1, Nz):
        A = _ssfm64(A, ea if idx[i] else en, nl)
        if not cp.isfinite(cp.sum(A)):
            raise RuntimeError(f"NaN/Inf at step {i}, z={i*dz:.3f} m")
        if i % rec == 0 or i == Nz-1:
            z_r.append(i*dz)
            e_r.append(_energy(A)/E0*100)
            p_r.append(_peak(A))
            f_r.append(_fwhm(A))
    return A, np.array(z_r), np.array(e_r), np.array(p_r), np.array(f_r)

def run64_ss4(dz, Ltot, P0, L1, L2, alpha=0.0):
    L_map = L1 + L2
    Nz    = int(np.round(Ltot/dz)) + 1
    A     = _pulse64(P0)
    for i in range(1, Nz):
        z_mid = (i-0.5)*dz
        beta2 = beta2_a if (z_mid % L_map >= L1) else beta2_n
        A = _ssfm64_ss4(A, beta2, dz, alpha)
        if not cp.isfinite(cp.sum(A)):
            raise RuntimeError(f"NaN/Inf at SS4 step {i}, z={i*dz:.3f} m")
    return A

# ==============================================================================
# 8. CONVERGENCE TEST ENGINE
# ==============================================================================
def convergence_test(Ltot, L1, L2, P0, dz_list, ref_dz,
                     pulse='gaussian', alpha=0.0, label="Case", **kw):
    print(f"\n[{label}]  Ltot={Ltot:.0f} m  L1={L1:.0f}  L2={L2:.0f}  "
          f"P0={P0:.2f} W  alpha={alpha:.2e}")
    t0 = time.time()
    A_ref = run64(ref_dz, make_indexer('midpoint'), Ltot, P0, L1, L2, pulse, alpha, **kw)
    print(f"  Ref (dz={ref_dz:.5f} m) built in {time.time()-t0:.1f} s")

    et, em, tt, tm = [], [], [], []
    for dh in tqdm(dz_list, ncols=72, desc=label):
        t0 = time.time()
        At = run64(dh, make_indexer('traditional'), Ltot, P0, L1, L2, pulse, alpha, **kw)
        tt.append(time.time() - t0)
        t0 = time.time()
        Am = run64(dh, make_indexer('midpoint'),    Ltot, P0, L1, L2, pulse, alpha, **kw)
        tm.append(time.time() - t0)
        et.append(_l2rel(At, A_ref))
        em.append(_l2rel(Am, A_ref))

    ot = [np.nan] + [np.log(et[i-1]/et[i])/np.log(dz_list[i-1]/dz_list[i])
                     for i in range(1, len(dz_list))]
    om = [np.nan] + [np.log(em[i-1]/em[i])/np.log(dz_list[i-1]/dz_list[i])
                     for i in range(1, len(dz_list))]
    avg_ot = np.nanmean(ot)
    avg_om = np.nanmean(om)

    print(f"\n  {'dz':>6} | {'Err Trad':>10} | {'p_T':>6} | {'Err Mid':>10} | {'p_M':>6} | {'Spdup':>6}")
    print(f"  {'-'*6}-|-{'-'*10}-|-{'-'*6}-|-{'-'*10}-|-{'-'*6}-|-{'-'*6}")
    for dh, e_t, o_t, e_m, o_m, t_t, t_m in zip(dz_list, et, ot, em, om, tt, tm):
        print(f"  {dh:6.3f} | {e_t:10.2e} | {o_t:6.3f} | {e_m:10.2e} | {o_m:6.3f} | {t_t/t_m:6.2f}")
    print(f"  => Trad avg order: {avg_ot:.3f}  |  Mid avg order: {avg_om:.3f}")
    if len(dz_list) > 1 and em[1] > 0:
        print(f"  => Error reduction @ dz={dz_list[1]}: {et[1]/em[1]:.0f}x")

    return dict(dz_list=dz_list, et=et, em=em, ot=ot, om=om,
                tt=tt, tm=tm, avg_ot=avg_ot, avg_om=avg_om,
                A_ref=A_ref, label=label)

# ==============================================================================
# 9. PLOTTING LIBRARY
# ==============================================================================
def _fit_slope(x, y, n=None):
    n = n or max(len(x)-1, 2)
    n = min(n, len(x))
    s, b, r, *_ = linregress(np.log10(x[:n]), np.log10(y[:n]))
    return s, b, r**2

def plot_convergence(data, fname, title=None, show_ss4=None):
    dz  = np.array(data['dz_list'])
    et  = np.array(data['et'])
    em  = np.array(data['em'])
    st, bt, _ = _fit_slope(dz, et)
    sm, bm, _ = _fit_slope(dz, em)
    dz_f   = np.logspace(np.log10(dz.min()*0.65), np.log10(dz.max()*1.5), 120)
    anchor = em[0] / dz[0]**2
    ref1   = anchor * dz_f**1
    ref2   = anchor * dz_f**2

    fig, ax = plt.subplots(figsize=(W1, H1*1.1))
    ax.loglog(dz, et, "s", c=C_TRAD, mfc="w", mec=C_TRAD, ms=5, mew=0.8,
              label="Traditional", zorder=5, clip_on=False)
    ax.loglog(dz_f, 10**bt*dz_f**st, "--", c=C_TRAD, lw=1.0,
              label=rf"Fit $p={st:.2f}$", zorder=4)
    ax.loglog(dz, em, "o", c=C_MID, mfc="w", mec=C_MID, ms=5, mew=0.8,
              label="Midpoint (this work)", zorder=5, clip_on=False)
    ax.loglog(dz_f, 10**bm*dz_f**sm, "-", c=C_MID, lw=1.0,
              label=rf"Fit $p={sm:.2f}$", zorder=4)
    if show_ss4 is not None:
        dz4 = np.array(show_ss4['dz_list'])
        e4  = np.array(show_ss4['errs'])
        s4, b4, _ = _fit_slope(dz4, e4)
        ax.loglog(dz4, e4, "D", c=C_SS4, mfc="w", mec=C_SS4, ms=5, mew=0.8,
                  label="SS4 (Yoshida)", zorder=5, clip_on=False)
        ax.loglog(dz_f, 10**b4*dz_f**s4, "-.", c=C_SS4, lw=1.0,
                  label=rf"Fit $p={s4:.2f}$", zorder=4)
        ax.loglog(dz_f, (e4[0]/dz4[0]**4)*dz_f**4, ":", c=C_REF4, lw=0.7,
                  label=r"$\mathcal{O}(\Delta z^4)$", zorder=2)
    ax.loglog(dz_f, ref1, ":", c=C_REF1, lw=0.7,
              label=r"$\mathcal{O}(\Delta z)$", zorder=2)
    ax.loglog(dz_f, ref2, ":", c=C_REF2, lw=0.7,
              label=r"$\mathcal{O}(\Delta z^2)$", zorder=2)
    ax.set_xlabel(r"Step size $\Delta z$ (m)")
    ax.set_ylabel(r"Relative $L_2$ error")
    if title:
        ax.set_title(title, pad=3)
    ax.legend(frameon=True, framealpha=0.9, edgecolor="0.8", ncol=2, handlelength=1.4)
    _grid(ax, log=True)
    _savefig(fig, figpath(fname))


def plot_local_error_both(z_trad, err_trad, z_mid, err_mid, z_interfaces, fname):
    fig, ax = plt.subplots(figsize=(W1, H1))
    ax.semilogy(z_trad, err_trad, c=C_TRAD, lw=1.0, label="Traditional")
    ax.semilogy(z_mid,  err_mid,  c=C_MID,  lw=1.0, label="Midpoint (this work)")
    for i, zif in enumerate(z_interfaces):
        ax.axvline(zif, c="0.5", lw=0.7, ls="--",
                   label=r"$\beta_2$ interface" if i == 0 else "_")
    ax.set_xlabel(r"Propagation distance $z$ (m)")
    ax.set_ylabel(r"Cumulative $L_2$ error")
    ax.legend(frameon=True, framealpha=0.9, edgecolor="0.8")
    _minor_ticks_linear(ax)
    _grid(ax)
    _savefig(fig, figpath(fname))


def plot_order_vs_ratio(ratios, orders_trad, orders_mid, fname):
    fig, ax = plt.subplots(figsize=(W1, H1))
    ax.semilogx(ratios, orders_trad, "s-", c=C_TRAD, ms=5, mfc="w", mec=C_TRAD,
                lw=1.0, label="Traditional")
    ax.semilogx(ratios, orders_mid,  "o-", c=C_MID,  ms=5, mfc="w", mec=C_MID,
                lw=1.0, label="Midpoint (this work)")
    ax.axhline(1.0, c=C_REF1, lw=0.7, ls=":", label=r"$p=1$ (1st-order)")
    ax.axhline(2.0, c=C_REF2, lw=0.7, ls=":", label=r"$p=2$ (2nd-order)")
    ax.set_xlabel(r"$L_\mathrm{map}/\Delta z$ (steps per map period)")
    ax.set_ylabel("Observed convergence order $p$")
    ax.set_ylim(-0.5, 2.5)
    ax.legend(frameon=True, framealpha=0.9, edgecolor="0.8")
    _minor_ticks_linear(ax)
    _grid(ax)
    _savefig(fig, figpath(fname))


def plot_ss4_comparison(data_mid, data_ss4, fname):
    fig, ax = plt.subplots(figsize=(W1, H1*1.1))
    ax.loglog(data_mid['tt'], data_mid['et'], "s--", c=C_TRAD, ms=5,
              mfc="w", mec=C_TRAD, lw=1.0, label="Trad (2nd-order SSFM)", alpha=0.7)
    ax.loglog(data_mid['tm'], data_mid['em'], "o-",  c=C_MID,  ms=5,
              mfc="w", mec=C_MID,  lw=1.0, label="Midpoint (this work, 2nd-order)")
    ax.loglog(data_ss4['times'], data_ss4['errs'], "D-.", c=C_SS4, ms=5,
              mfc="w", mec=C_SS4,  lw=1.0, label="SS4/Yoshida (4th-order)")
    ax.set_xlabel("Wall-clock time (s)")
    ax.set_ylabel(r"Relative $L_2$ error")
    ax.legend(frameon=True, framealpha=0.9, edgecolor="0.8", handlelength=1.4)
    _grid(ax, log=True)
    _savefig(fig, figpath(fname))


def plot_efficiency_pareto(all_data, fname):
    colors = [C_TRAD, C_MID, C_SS4, C_REF4, "#E69F00", "#56B4E9"]
    fig, ax = plt.subplots(figsize=(W1, H1*1.1))
    for k, d in enumerate(all_data):
        c   = colors[k % len(colors)]
        lbl = d.get('label', f"Case {k+1}")
        ax.loglog(d['tt'], d['et'], "s--", c=c, ms=4, lw=0.8, alpha=0.55, mfc="w", mec=c)
        ax.loglog(d['tm'], d['em'], "o-",  c=c, ms=4, lw=0.8, mfc="w", mec=c, label=lbl)
    from matplotlib.lines import Line2D
    h, l = ax.get_legend_handles_labels()
    extras = [
        Line2D([0],[0], ls="--", c="0.4", lw=0.8, marker="s", ms=4, mfc="w", label="Traditional"),
        Line2D([0],[0], ls="-",  c="0.4", lw=0.8, marker="o", ms=4, mfc="w", label="Midpoint"),
    ]
    ax.legend(handles=h+extras, frameon=True, framealpha=0.9, edgecolor="0.8",
              ncol=2, handlelength=1.4)
    ax.set_xlabel("Wall-clock time (s)")
    ax.set_ylabel(r"Relative $L_2$ error")
    _grid(ax, log=True)
    _savefig(fig, figpath(fname))


# ==============================================================================
# 9b. PANEL PLOTS
# ==============================================================================
def _add_panel_label(ax, label, x=-0.18, y=1.05):
    ax.text(x, y, f"({label})", transform=ax.transAxes,
            fontsize=10, fontweight='bold', va='top', ha='right')


def plot_panel_case1(data, zr_t, etr, ptr, zr_m, emr, pmr, fname):
    """
    Case 1: 1x3 panel.
    [FIX 6] Separate x-arrays zr_t / zr_m for traditional / midpoint
            in panels (b) and (c) to avoid silent length mismatch.
    """
    dz  = np.array(data['dz_list'])
    et  = np.array(data['et'])
    em  = np.array(data['em'])
    st, bt, _ = _fit_slope(dz, et)
    sm, bm, _ = _fit_slope(dz, em)
    dz_f   = np.logspace(np.log10(dz.min()*0.65), np.log10(dz.max()*1.5), 120)
    anchor = em[0] / dz[0]**2
    ref1   = anchor * dz_f**1
    ref2   = anchor * dz_f**2

    fig, axes = plt.subplots(1, 3, figsize=(W2, W2/3.2))

    ax = axes[0]
    ax.loglog(dz, et, "s", c=C_TRAD, mfc="w", mec=C_TRAD, ms=5, mew=0.8,
              label="Traditional", zorder=5, clip_on=False)
    ax.loglog(dz_f, 10**bt*dz_f**st, "--", c=C_TRAD, lw=1.0,
              label=rf"Fit $p={st:.2f}$", zorder=4)
    ax.loglog(dz, em, "o", c=C_MID, mfc="w", mec=C_MID, ms=5, mew=0.8,
              label="Midpoint (this work)", zorder=5, clip_on=False)
    ax.loglog(dz_f, 10**bm*dz_f**sm, "-", c=C_MID, lw=1.0,
              label=rf"Fit $p={sm:.2f}$", zorder=4)
    ax.loglog(dz_f, ref1, ":", c=C_REF1, lw=0.7, label=r"$\mathcal{O}(\Delta z)$", zorder=2)
    ax.loglog(dz_f, ref2, ":", c=C_REF2, lw=0.7, label=r"$\mathcal{O}(\Delta z^2)$", zorder=2)
    ax.set_xlabel(r"Step size $\Delta z$ (m)")
    ax.set_ylabel(r"Relative $L_2$ error")
    ax.legend(frameon=True, framealpha=0.9, edgecolor="0.8", ncol=1,
              handlelength=1.4, fontsize=6.5)
    _grid(ax, log=True)
    _add_panel_label(ax, "a")

    ax = axes[1]
    ax.plot(zr_t, etr, c=C_TRAD, lw=1.0, label="Traditional")
    ax.plot(zr_m, emr, c=C_MID,  lw=1.0, label="Midpoint")   # [FIX 6]
    ax.axhline(100, c="0.5", lw=0.6, ls=":")
    ax.set_xlabel(r"$z$ (m)")
    ax.set_ylabel("Energy retention (%)")
    ax.set_ylim(99.9, 100.1)
    ax.legend(frameon=True, framealpha=0.9, edgecolor="0.8", fontsize=6.5)
    _minor_ticks_linear(ax)
    _grid(ax)
    _add_panel_label(ax, "b")

    ax = axes[2]
    ax.plot(zr_t, ptr, c=C_TRAD, lw=1.0, label="Traditional")
    ax.plot(zr_m, pmr, c=C_MID,  lw=1.0, label="Midpoint")   # [FIX 6]
    ax.set_xlabel(r"$z$ (m)")
    ax.set_ylabel("Peak power (W)")
    ax.legend(frameon=True, framealpha=0.9, edgecolor="0.8", fontsize=6.5)
    _minor_ticks_linear(ax)
    _grid(ax)
    _add_panel_label(ax, "c")

    fig.tight_layout(pad=0.5)
    fig.savefig(figpath(fname), format="pdf")
    print(f"[PANEL] -> {figpath(fname)}")
    plt.close(fig)


def plot_panel_case7(zr, etr, emr, ptr, zr2, pmr, ftr, fmr, fname):
    fig, axes = plt.subplots(1, 3, figsize=(W2, W2/3.2))
    ax = axes[0]
    ax.plot(zr/1000, etr, c=C_TRAD, lw=1.0, label="Traditional")
    ax.plot(zr/1000, emr, c=C_MID,  lw=1.0, label="Midpoint")
    ax.axhline(100, c="0.5", lw=0.6, ls=":")
    ax.set_xlabel(r"$z$ (km)"); ax.set_ylabel("Energy retention (%)")
    ax.set_ylim(99.5, 100.5)
    ax.legend(frameon=True, framealpha=0.9, edgecolor="0.8", fontsize=6.5)
    _minor_ticks_linear(ax); _grid(ax); _add_panel_label(ax, "a")

    ax = axes[1]
    ax.plot(zr/1000, ptr, c=C_TRAD, lw=1.0, label="Traditional")
    ax.plot(zr2/1000, pmr, c=C_MID, lw=1.0, label="Midpoint")
    ax.set_xlabel(r"$z$ (km)"); ax.set_ylabel("Peak power (W)")
    ax.legend(frameon=True, framealpha=0.9, edgecolor="0.8", fontsize=6.5)
    _minor_ticks_linear(ax); _grid(ax); _add_panel_label(ax, "b")

    ax = axes[2]
    ax.plot(zr/1000,  ftr, c=C_TRAD, lw=1.0, label="Traditional")
    ax.plot(zr2/1000, fmr, c=C_MID,  lw=1.0, label="Midpoint")
    ax.set_xlabel(r"$z$ (km)"); ax.set_ylabel(r"FWHM / $T_0$")
    ax.legend(frameon=True, framealpha=0.9, edgecolor="0.8", fontsize=6.5)
    _minor_ticks_linear(ax); _grid(ax); _add_panel_label(ax, "c")

    fig.tight_layout(pad=0.5)
    fig.savefig(figpath(fname), format="pdf")
    print(f"[PANEL] -> {figpath(fname)}")
    plt.close(fig)


def _conv_panel_ax(ax, data, idx, show_legend=False):
    dz = np.array(data['dz_list'])
    et = np.array(data['et'])
    em = np.array(data['em'])
    st, bt, _ = _fit_slope(dz, et)
    sm, bm, _ = _fit_slope(dz, em)
    dz_f   = np.logspace(np.log10(dz.min()*0.65), np.log10(dz.max()*1.5), 120)
    anchor = em[0] / dz[0]**2
    ax.loglog(dz, et, "s", c=C_TRAD, mfc="w", mec=C_TRAD, ms=5, mew=0.8,
              label="Traditional", zorder=5, clip_on=False)
    ax.loglog(dz_f, 10**bt*dz_f**st, "--", c=C_TRAD, lw=1.0, zorder=4)
    ax.loglog(dz, em, "o", c=C_MID, mfc="w", mec=C_MID, ms=5, mew=0.8,
              label="Midpoint", zorder=5, clip_on=False)
    ax.loglog(dz_f, 10**bm*dz_f**sm, "-", c=C_MID, lw=1.0, zorder=4)
    ax.loglog(dz_f, anchor*dz_f**1, ":", c=C_REF1, lw=0.7, zorder=2)
    ax.loglog(dz_f, anchor*dz_f**2, ":", c=C_REF2, lw=0.7, zorder=2)
    ax.set_xlabel(r"Step size $\Delta z$ (m)")
    ax.set_ylabel(r"Relative $L_2$ error")
    if show_legend:
        ax.legend(frameon=True, framealpha=0.9, edgecolor="0.8",
                  ncol=1, handlelength=1.4, fontsize=6.5)
    _grid(ax, log=True)
    _add_panel_label(ax, chr(ord('a')+idx))


def plot_panel_cases234(data_list, titles, fname):
    fig, axes = plt.subplots(1, 3, figsize=(W2, W2/3.2))
    for i, (data, ttl) in enumerate(zip(data_list, titles)):
        _conv_panel_ax(axes[i], data, i, show_legend=(i==2))
        axes[i].set_title(ttl, pad=3, fontsize=8)
    fig.tight_layout(pad=0.5)
    fig.savefig(figpath(fname), format="pdf")
    print(f"[PANEL] -> {figpath(fname)}")
    plt.close(fig)


def plot_panel_cases1011(data10, data11, fname):
    titles = ["Sech pulse", r"Lossy fiber, $\alpha=0.2$ dB/km"]
    fig, axes = plt.subplots(1, 2, figsize=(W2*0.67, W2*0.67/3.2))
    for i, (data, ttl) in enumerate(zip([data10, data11], titles)):
        _conv_panel_ax(axes[i], data, i, show_legend=(i==1))
        axes[i].set_title(ttl, pad=3, fontsize=8)
    fig.tight_layout(pad=0.5)
    fig.savefig(figpath(fname), format="pdf")
    print(f"[PANEL] -> {figpath(fname)}")
    plt.close(fig)


def plot_panel_case12(delta_list, err_trad, err_mid, fit_trad, fit_mid,
                      r2_trad, r2_mid, fname):
    """
    Case 12: 1x2 panel -- Proposition 1 verification (K=10 interface sweep).

    Left:  Traditional error vs delta with (1-delta) fit.
           Step-start eval at j=4 always assigns N-fiber; the fraction of
           step j=4 that should be A-fiber is (1-delta), giving error ∝ (1-delta).
    Right: Midpoint error vs delta with min(d,1-d) fit.
           Step-centre eval at j=4 (z=45 m) assigns A when L1_eff<45 (delta<0.5)
           and N when L1_eff>45 (delta>0.5); error ∝ min(delta, 1-delta).
    Both fits achieve R²≈1.000, confirming Proposition 1.
    """
    fig, axes = plt.subplots(1, 2, figsize=(W2, H1))

    ax = axes[0]
    ax.plot(delta_list, err_trad, "s", c=C_TRAD, mfc="w", mec=C_TRAD,
            ms=4, mew=0.7, label="Traditional (K=10)", zorder=5)
    ax.plot(delta_list, fit_trad, "-", c=C_TRAD, lw=1.2,
            label=rf"Fit $\propto(1-\delta)$  ($R^2={r2_trad:.4f}$)",
            zorder=4)
    ax.set_xlabel(r"Fractional offset $\delta$")
    ax.set_ylabel(r"Relative $L_2$ error")
    ax.set_title(r"Traditional: error $\propto(1-\delta)$", fontsize=8.5)
    ax.legend(frameon=True, framealpha=0.9, edgecolor="0.8", fontsize=6.5)
    _minor_ticks_linear(ax); _grid(ax); _add_panel_label(ax, "a")

    ax = axes[1]
    ax.plot(delta_list, err_mid, "o", c=C_MID, mfc="w", mec=C_MID,
            ms=4, mew=0.7, label="Midpoint (K=10)", zorder=5)
    ax.plot(delta_list, fit_mid, "-", c=C_MID, lw=1.2,
            label=rf"Fit $\propto\min(\delta,1\!-\!\delta)$  ($R^2={r2_mid:.4f}$)",
            zorder=4)
    ax.set_xlabel(r"Fractional offset $\delta$")
    ax.set_ylabel(r"Relative $L_2$ error")
    ax.set_title(r"Midpoint: error $\propto\min(\delta,1-\delta)$", fontsize=8.5)
    ax.legend(frameon=True, framealpha=0.9, edgecolor="0.8", fontsize=6.5)
    _minor_ticks_linear(ax); _grid(ax); _add_panel_label(ax, "b")

    fig.tight_layout(pad=0.5)
    fig.savefig(figpath(fname), format="pdf")
    print(f"[PANEL] -> {figpath(fname)}")
    plt.close(fig)


# ==============================================================================
# MOMENTUM DEVIATION HELPER
# ==============================================================================
def _momentum_dev(A):
    A_fft  = cp.fft.fft(A)
    S      = cp.abs(A_fft)**2
    E_spec = cp.sum(S)
    if E_spec < 1e-30:
        return 0.0
    return float(cp.abs(cp.sum(w_64 * S) / E_spec))

# ==============================================================================
# DATA OUTPUT DIRECTORY + STORAGE
# ==============================================================================
RESULTS_DIR = "./results_v6"
os.makedirs(RESULTS_DIR, exist_ok=True)
case_data     = {}
results_store = {}

# ==============================================================================
# 10. CASE RUNNERS
# ==============================================================================

def run_case_0():
    print(sep); print("CASE 0: REFERENCE SELF-CONSISTENCY CHECK"); print(sep_s)
    L1 = L2 = 50.0; Ltot = 6000.0; P0 = 0.30
    ref_dz  = dz_base / 32
    ref2_dz = ref_dz / 2

    t0 = time.time()
    A_ref1 = run64(ref_dz,  make_indexer('midpoint'), Ltot, P0, L1, L2)
    print(f"  Ref1  (dz={ref_dz:.5f} m) built in {time.time()-t0:.1f} s")
    t0 = time.time()
    A_ref2 = run64(ref2_dz, make_indexer('midpoint'), Ltot, P0, L1, L2)
    print(f"  Ref2  (dz={ref2_dz:.5f} m) built in {time.time()-t0:.1f} s")

    err_ref_self  = _l2rel(A_ref1, A_ref2)
    At = run64(1.0, make_indexer('traditional'), Ltot, P0, L1, L2)
    Am = run64(1.0, make_indexer('midpoint'),    Ltot, P0, L1, L2)
    err_test_trad = _l2rel(At, A_ref2)
    err_test_mid  = _l2rel(Am, A_ref2)
    ratio_trad    = err_test_trad / max(err_ref_self, 1e-30)
    ratio_mid     = err_test_mid  / max(err_ref_self, 1e-30)

    print(f"\n  Ref self-error:       {err_ref_self:.3e}")
    print(f"  Trad error (dz=1.0):  {err_test_trad:.3e}   (ratio: {ratio_trad:.0f}x)")
    print(f"  Mid  error (dz=1.0):  {err_test_mid:.3e}   (ratio: {ratio_mid:.0f}x)")
    print(f"  => Ref error is {ratio_trad:.0f}x smaller than trad -- safe as reference.")
    case_data['case0'] = dict(
        err_ref_self=np.float64(err_ref_self),
        err_test_trad=np.float64(err_test_trad),
        err_test_mid=np.float64(err_test_mid),
        ratio_trad=np.float64(ratio_trad),
        ratio_mid=np.float64(ratio_mid))
    print(sep)


def run_case_1():
    print(sep); print("CASE 1: BASELINE SYMMETRIC MAP  N≈1.4"); print(sep_s)
    L1 = L2 = 50.0; Ltot = 6000.0; P0 = 0.30
    dz_list = [2.0, 1.0, 0.5, 0.25]
    ref_dz  = dz_base / 32
    d = convergence_test(Ltot, L1, L2, P0, dz_list, ref_dz, label="Case 1 (Baseline)")
    results_store['case1'] = d

    dz_t = 1.0
    _, zr_t, etr, ptr, _ = run64_diag(dz_t, make_indexer('traditional'), Ltot, P0, L1, L2, rec=100)
    _, zr_m, emr, pmr, _ = run64_diag(dz_t, make_indexer('midpoint'),    Ltot, P0, L1, L2, rec=100)
    plot_panel_case1(d, zr_t, etr, ptr, zr_m, emr, pmr, "Fig01_Case1_Baseline_Panel.pdf")
    print(sep)


def run_case_2():
    print(sep); print("CASE 2: STRONG BREATHING  P0=0.60 W  N≈2.0"); print(sep_s)
    L1 = L2 = 50.0; Ltot = 2000.0; P0 = 0.60
    d = convergence_test(Ltot, L1, L2, P0, [1.0, 0.5, 0.25], dz_base/16, label="Case 2 (Strong)")
    results_store['case2'] = d
    print(sep)


def run_case_3():
    print(sep); print("CASE 3: ASYMMETRIC MAP  60:40 DUTY CYCLE"); print(sep_s)
    L1, L2 = 60.0, 40.0; Ltot = 2000.0; P0 = 0.30
    d = convergence_test(Ltot, L1, L2, P0, [1.0, 0.5, 0.25], dz_base/16, label="Case 3 (60:40)")
    results_store['case3'] = d
    print(sep)


def run_case_4():
    print(sep); print("CASE 4: TWO-SOLITON COLLISION  phase=π  sep=6T0"); print(sep_s)
    L1 = L2 = 50.0; Ltot = 2000.0; P0 = 0.30
    d = convergence_test(Ltot, L1, L2, P0, [1.0, 0.5, 0.25], dz_base/16,
                         pulse='two', label="Case 4 (Two-soliton)",
                         separation=6.0, phase_diff=np.pi)
    results_store['case4'] = d
    print(sep)


def run_case_5():
    print(sep); print("CASE 5: LOCAL ERROR MICROSCOPE -- Both Methods, 3 Map Periods"); print(sep_s)
    L1 = L2 = 50.0; Ltot = 300.0; P0 = 0.30
    dz_test = 0.5; dz_ref = 0.005

    ratio  = int(np.round(dz_test / dz_ref))
    Nz_ref = int(np.round(Ltot / dz_ref)) + 1
    nl_r   = _nl64(dz_ref)
    A_r    = _pulse64(P0)
    snaps, z_snaps = [], []
    for i in range(1, Nz_ref):
        z_mid_ref = (i - 0.5) * dz_ref
        b2 = beta2_a if (z_mid_ref % (L1 + L2) >= L1) else beta2_n
        A_r = _ssfm64(A_r, _exp64(b2, dz_ref), nl_r)
        if i % ratio == 0:
            snaps.append(A_r.copy())
            z_snaps.append(i * dz_ref)

    local = {}
    for mname, meth in [("Traditional", 'traditional'), ("Midpoint", 'midpoint')]:
        Nz, idx = build_index(Ltot, dz_test, L1, L2, meth)
        en = _exp64(beta2_n, dz_test); ea = _exp64(beta2_a, dz_test)
        nl = _nl64(dz_test); A = _pulse64(P0)
        errs, zs = [], []
        for i in range(1, Nz):
            A = _ssfm64(A, ea if idx[i] else en, nl)
            si = i - 1
            if si < len(snaps):
                errs.append(_l2rel(A, snaps[si])); zs.append(i * dz_test)
        local[mname] = (np.array(zs), np.array(errs))
        print(f"  [{mname}] max={max(errs):.2e}  mean={np.mean(errs):.2e}  "
              f"ratio(max/mean)={max(errs)/np.mean(errs):.2f}")

    z_iface = [z for z in [L1, L1+L2, 2*L1+L2, 2*(L1+L2), 2*L1+2*L2] if z <= Ltot]
    plot_local_error_both(local["Traditional"][0], local["Traditional"][1],
                          local["Midpoint"][0],    local["Midpoint"][1],
                          z_iface, "Fig05_Case5_LocalError_BothMethods.pdf")
    case_data['case5'] = dict(
        z_t=local["Traditional"][0], e_t=local["Traditional"][1],
        z_m=local["Midpoint"][0],    e_m=local["Midpoint"][1],
        z_iface=np.array(z_iface))
    print(sep)


def run_case_6():
    print(sep); print("CASE 6: EFFICIENCY PARETO  L=10 km"); print(sep_s)
    L1 = L2 = 50.0; Ltot = 10000.0; P0 = 0.30
    d = convergence_test(Ltot, L1, L2, P0, [2.0, 1.0, 0.5, 0.25, 0.125],
                         dz_base/8, label="Case 6 (Efficiency)")
    results_store['case6'] = d
    print(sep)


def run_case_7():
    print(sep); print("CASE 7: LONG-HAUL 50 km -- Energy + FWHM + Peak + Momentum"); print(sep_s)
    L1 = L2 = 50.0; Ltot = 50000.0; P0 = 0.30; dz = 1.0
    results_7 = {}
    for mname, meth in [("Traditional", 'traditional'), ("Midpoint", 'midpoint')]:
        Nz, idx = build_index(Ltot, dz, L1, L2, meth)
        en = _exp64(beta2_n, dz); ea = _exp64(beta2_a, dz); nl = _nl64(dz)
        A  = _pulse64(P0); E0 = _energy(A); mom0 = _momentum_dev(A)
        z_r, e_r, p_r, f_r, m_r = [], [], [], [], []
        rec = 500
        for i in range(1, Nz):
            A = _ssfm64(A, ea if idx[i] else en, nl)
            if not cp.isfinite(cp.sum(A)):
                raise RuntimeError(f"NaN/Inf at step {i}")
            if i % rec == 0 or i == Nz-1:
                z_r.append(i*dz); e_r.append(_energy(A)/E0*100)
                p_r.append(_peak(A)); f_r.append(_fwhm(A))
                m_r.append(abs(_momentum_dev(A) - mom0))
        tag = 't' if meth == 'traditional' else 'm'
        for k, v in zip(['z','E','P','F','M'], [z_r,e_r,p_r,f_r,m_r]):
            results_7[f'{k}_{tag}'] = np.array(v)
        print(f"  {mname}: E_final={e_r[-1]:.6f}%  "
              f"peak_drift={(p_r[-1]-p_r[0])/p_r[0]*100:.3f}%  "
              f"mom_dev_final={m_r[-1]:.4e}")
    case_data['case7'] = results_7
    pk_diff = abs((results_7['P_m'][-1]-results_7['P_m'][0])/results_7['P_m'][0] -
                  (results_7['P_t'][-1]-results_7['P_t'][0])/results_7['P_t'][0])
    print(f"  Peak-drift difference = {pk_diff*100:.3f} pp")
    plot_panel_case7(results_7['z_t'], results_7['E_t'], results_7['E_m'],
                     results_7['P_t'], results_7['z_m'], results_7['P_m'],
                     results_7['F_t'], results_7['F_m'],
                     "Fig07_Case7_LongHaul_Panel.pdf")
    print(sep)


def run_case_8():
    print(sep); print("CASE 8: STEP-RATIO ROBUSTNESS -- order vs L_map/dz"); print(sep_s)
    L1 = L2 = 50.0; L_map = L1+L2; Ltot = 2000.0; P0 = 0.30
    ratios_target = [10, 20, 40, 80, 160, 320, 640]
    ratios_actual, orders_trad, orders_mid = [], [], []
    for r in tqdm(ratios_target, ncols=72, desc="Case 8"):
        dz_c = L_map/r; dz_f = dz_c/2; dz_ff = dz_f/2
        A_ref = run64(dz_ff/8, make_indexer('midpoint'), Ltot, P0, L1, L2)
        errors = {}
        for meth in ('traditional', 'midpoint'):
            ix = make_indexer(meth)
            errors[meth] = [_l2rel(run64(dz, ix, Ltot, P0, L1, L2), A_ref)
                            for dz in (dz_c, dz_f, dz_ff)]
        ot = np.mean([np.log(errors['traditional'][i-1]/errors['traditional'][i])/np.log(2)
                      for i in range(1,3)])
        om = np.mean([np.log(errors['midpoint'][i-1]/errors['midpoint'][i])/np.log(2)
                      for i in range(1,3)])
        ratios_actual.append(r); orders_trad.append(ot); orders_mid.append(om)
        note = "  <-- float64 noise floor" if om < 1.0 else ""
        print(f"  ratio={r:4d}  dz_coarse={dz_c:.4f} m  "
              f"order_trad={ot:.3f}  order_mid={om:.3f}{note}")
    plot_order_vs_ratio(ratios_actual, orders_trad, orders_mid,
                        "Fig08_Case8_StepRatioRobustness.pdf")
    results_store['case8'] = dict(ratios=ratios_actual,
                                  orders_trad=orders_trad, orders_mid=orders_mid)
    print(sep)


def run_case_9():
    print(sep); print("CASE 9: HIGHER-ORDER COMPARISON -- Midpoint-2nd vs SS4 (Yoshida)"); print(sep_s)
    L1 = L2 = 50.0; Ltot = 2000.0; P0 = 0.30
    dz_list_23  = [2.0, 1.0, 0.5, 0.25, 0.125]
    dz_list_ss4 = [4.0, 2.0, 1.0, 0.5, 0.25]
    A_ref = run64(dz_base/16, make_indexer('midpoint'), Ltot, P0, L1, L2)

    et, em, tt, tm = [], [], [], []
    for dh in tqdm(dz_list_23, ncols=72, desc="Case 9 (2nd-order)"):
        t0 = time.time()
        At = run64(dh, make_indexer('traditional'), Ltot, P0, L1, L2)
        tt.append(time.time()-t0)
        t0 = time.time()
        Am = run64(dh, make_indexer('midpoint'), Ltot, P0, L1, L2)
        tm.append(time.time()-t0)
        et.append(_l2rel(At, A_ref)); em.append(_l2rel(Am, A_ref))

    times_ss4, errs_ss4 = [], []
    for dh in tqdm(dz_list_ss4, ncols=72, desc="Case 9 (SS4)"):
        t0 = time.time()
        A4 = run64_ss4(dh, Ltot, P0, L1, L2)
        times_ss4.append(time.time()-t0); errs_ss4.append(_l2rel(A4, A_ref))

    print("\n  2nd-order results:")
    for dh, e_t, e_m, t_t, t_m in zip(dz_list_23, et, em, tt, tm):
        print(f"    dz={dh:.3f}  Trad={e_t:.2e} ({t_t:.2f}s)  "
              f"Mid={e_m:.2e} ({t_m:.2f}s)  ratio={e_t/e_m:.0f}x")
    print("\n  SS4 results (4th-order; floor reached within 1-2 halvings):")
    for dh, e4, t4 in zip(dz_list_ss4, errs_ss4, times_ss4):
        print(f"    dz={dh:.3f}  SS4={e4:.2e} ({t4:.2f}s)")

    data2nd = dict(tt=tt, tm=tm, et=et, em=em, label="2nd-order SSFM")
    plot_ss4_comparison(data2nd, dict(times=times_ss4, errs=errs_ss4),
                        "Fig09_Case9_SS4_Comparison.pdf")
    results_store['case9_2nd'] = data2nd
    case_data['case9'] = dict(time_trad=np.array(tt), err_trad=np.array(et),
                               time_mid=np.array(tm),  err_mid=np.array(em),
                               time_ss4=np.array(times_ss4), err_ss4=np.array(errs_ss4))
    print(sep)


def run_case_10():
    print(sep); print("CASE 10: INITIAL CONDITION -- sech pulse (non-Gaussian)"); print(sep_s)
    L1 = L2 = 50.0; Ltot = 2000.0; P0 = 0.30
    d = convergence_test(Ltot, L1, L2, P0, [1.0, 0.5, 0.25], dz_base/16,
                         pulse='sech', label="Case 10 (sech IC)")
    results_store['case10'] = d
    print(sep)


def run_case_11():
    print(sep); print("CASE 11: LOSSY FIBER  alpha=0.2 dB/km"); print(sep_s)
    alpha_dBkm = 0.2
    alpha_pm   = alpha_dBkm * np.log(10) / (10 * 1000)
    print(f"  alpha = {alpha_dBkm} dB/km = {alpha_pm:.4e} Np/m")
    L1 = L2 = 50.0; Ltot = 2000.0; P0 = 0.30
    d = convergence_test(Ltot, L1, L2, P0, [1.0, 0.5, 0.25], dz_base/16,
                         alpha=alpha_pm, label="Case 11 (Lossy)")
    results_store['case11'] = d

    lossy_diag = {}
    for mname, meth in [("Traditional", 'traditional'), ("Midpoint", 'midpoint')]:
        dz_diag = 1.0
        Nz, idx = build_index(Ltot, dz_diag, L1, L2, meth)
        en = _exp64(beta2_n, dz_diag, alpha_pm)
        ea = _exp64(beta2_a, dz_diag, alpha_pm)
        nl = _nl64(dz_diag); A = _pulse64(P0); E0 = _energy(A)
        z_all, dev_all = [], []
        for i in range(1, Nz):
            A = _ssfm64(A, ea if idx[i] else en, nl)
            if i % 20 == 0 or i == Nz-1:
                z_cur = i * dz_diag
                E_cur = _energy(A)
                # [FIX 2] _energy = integral |A|^2 dt.  With correct alpha
                # factor, |A| decays as exp(-alpha*z/2), so |A|^2 ~ exp(-alpha*z).
                E_expected = E0 * np.exp(-alpha_pm * z_cur)
                dev_all.append(abs(E_cur - E_expected)); z_all.append(z_cur)
        tag = 't' if meth == 'traditional' else 'm'
        lossy_diag[f'z_{tag}']   = np.array(z_all)
        lossy_diag[f'dev_{tag}'] = np.array(dev_all)
    case_data['case11'] = lossy_diag
    print(sep)


def run_case_12():
    """
    Case 12: Proposition 1 verification via physical interface sweep.

    Setup: K=10 steps, dz=10 m, L_map=100 m, N_maps=20 (Ltot=2000 m).
    Interface position: L1_eff = 40 + delta*10 (inside step j=4, spans [40,50)m).
    Reference: EXACT ANALYTIC solution for the SAME L1_eff (gamma=0, pure dispersion).

    Traditional (eval at step START = j*dz):
      j=4: eval = 40 < L1_eff for all delta>0 → always assigns N-fiber.
      Fraction of step j=4 that should be A-fiber = (50-L1_eff)/dz = 1-delta.
      Error ∝ (1-delta).  R²≈1.000.

    Midpoint (eval at step CENTRE = (j+0.5)*dz = 45):
      delta < 0.5: L1_eff < 45 → assigns A-fiber; error ∝ delta (wrong N fraction).
      delta > 0.5: L1_eff > 45 → assigns N-fiber; error ∝ (1-delta) (wrong A fraction).
      Error ∝ min(delta, 1-delta).  R²≈1.000.

    Key insight: Traditional is ALWAYS worse than or equal to Midpoint.
    At delta=0.5 (interface at step centre): both have equal (0.5-fraction) error.
    """
    print(sep)
    print("CASE 12: INTERFACE SWEEP -- Proposition 1 verification (gamma=0)")
    print(sep_s)

    P0 = 0.30; L_map = 100.0; K = 10
    dz = L_map / K          # = 10.0 m exactly
    N_maps = 20; Ltot = N_maps * L_map   # = 2000 m

    print(f"  K={K}, dz={dz:.1f} m, L_map={L_map:.0f} m, N_maps={N_maps}, Ltot={Ltot:.0f} m")
    print(f"  L1_eff = 40 + delta*{dz:.0f}  (interface inside step j=4, spans [40,50)m)")
    print(f"  Traditional j=4 eval=40 < L1_eff always → error ∝ (1-delta)")
    print(f"  Midpoint    j=4 eval=45: correct for delta<0.5, wrong for delta>0.5")
    print(f"              → error ∝ min(d,1-d)")

    A0_gpu = _pulse64(P0)
    A0_fft = cp.fft.fft(A0_gpu)

    delta_list    = np.linspace(0.02, 0.98, 49)
    err_trad_list = []
    err_mid_list  = []

    for delta in tqdm(delta_list, ncols=72, desc="Case 12"):
        L1_eff = 40.0 + delta * dz
        L2_eff = L_map - L1_eff

        # Exact analytic reference (pure dispersion, frequency-domain product)
        A_fft_ref = A0_fft.copy()
        for _ in range(N_maps):
            A_fft_ref = A_fft_ref * cp.exp(0.5j * beta2_n * w_64**2 * L1_eff)
            A_fft_ref = A_fft_ref * cp.exp(0.5j * beta2_a * w_64**2 * L2_eff)
        A_ref = cp.fft.ifft(A_fft_ref)

        # Traditional: eval at step START = j*dz
        A_fft_t = A0_fft.copy()
        for n in range(N_maps):
            z0 = n * L_map
            for j in range(K):
                z_eval = z0 + j * dz
                b2 = beta2_a if (z_eval % L_map >= L1_eff) else beta2_n
                A_fft_t = A_fft_t * cp.exp(0.5j * b2 * w_64**2 * dz)
        A_t = cp.fft.ifft(A_fft_t)

        # Midpoint: eval at step CENTRE = (j+0.5)*dz
        A_fft_m = A0_fft.copy()
        for n in range(N_maps):
            z0 = n * L_map
            for j in range(K):
                z_eval = z0 + (j + 0.5) * dz
                b2 = beta2_a if (z_eval % L_map >= L1_eff) else beta2_n
                A_fft_m = A_fft_m * cp.exp(0.5j * b2 * w_64**2 * dz)
        A_m = cp.fft.ifft(A_fft_m)

        err_trad_list.append(_l2rel(A_t, A_ref))
        err_mid_list.append(_l2rel(A_m, A_ref))

    err_trad = np.array(err_trad_list)
    err_mid  = np.array(err_mid_list)

    # Fit Traditional: error ∝ (1-delta)
    basis_t  = 1.0 - delta_list
    amp_t    = float(np.sum(basis_t * err_trad) / max(np.sum(basis_t**2), 1e-30))
    fit_t    = amp_t * basis_t
    ss_res   = np.sum((err_trad - fit_t)**2)
    ss_tot   = np.sum((err_trad - np.mean(err_trad))**2)
    r2_t     = 1.0 - ss_res / max(ss_tot, 1e-30)

    # Fit Midpoint: error ∝ min(delta, 1-delta)
    basis_m  = np.minimum(delta_list, 1.0 - delta_list)
    amp_m    = float(np.sum(basis_m * err_mid) / max(np.sum(basis_m**2), 1e-30))
    fit_m    = amp_m * basis_m
    ss_res   = np.sum((err_mid - fit_m)**2)
    ss_tot   = np.sum((err_mid - np.mean(err_mid))**2)
    r2_m     = 1.0 - ss_res / max(ss_tot, 1e-30)

    print(f"\n  Traditional: amp={amp_t:.4e},  R²={r2_t:.6f}")
    print(f"  Midpoint:    amp={amp_m:.4e},  R²={r2_m:.6f}")
    print(f"  Trad range:  [{err_trad.min():.3e}, {err_trad.max():.3e}]")
    print(f"  Mid  range:  [{err_mid.min():.3e}, {err_mid.max():.3e}]"
          f"  (min at delta≈0.5: {err_mid[24]:.3e})")

    np.savez_compressed(os.path.join(RESULTS_DIR, "case12.npz"),
                        delta_list=delta_list,
                        err_trad=err_trad, err_mid=err_mid,
                        fit_trad=fit_t, fit_mid=fit_m,
                        amp_trad=np.float64(amp_t), amp_mid=np.float64(amp_m),
                        r2_trad=np.float64(r2_t), r2_mid=np.float64(r2_m),
                        K=np.int64(K), dz=np.float64(dz),
                        N_maps=np.int64(N_maps))
    print(f"  [EXPORT] -> {RESULTS_DIR}/case12.npz")

    plot_panel_case12(delta_list, err_trad, err_mid, fit_t, fit_m,
                      r2_t, r2_m,
                      "Fig12_Case12_DeltaSweep_Panel.pdf")
    print(sep)


# ==============================================================================
# EXPORT ALL RESULTS TO .npz
# ==============================================================================
def export_all_results():
    print(sep); print("EXPORTING ALL RESULTS TO .npz"); print(sep_s)
    n_saved = 0

    if 'case0' in case_data:
        np.savez_compressed(os.path.join(RESULTS_DIR, "case0.npz"), **case_data['case0'])
        print("  [EXPORT] case0.npz"); n_saved += 1

    # [FIX 4] Case10, Case11 now included
    conv_map = {'case1':'Case1','case2':'Case2','case3':'Case3','case4':'Case4',
                'case6':'Case6','case10':'Case10','case11':'Case11'}
    for key, label in conv_map.items():
        if key not in results_store: continue
        d = results_store[key]
        np.savez_compressed(os.path.join(RESULTS_DIR, f"{label}_conv.npz"),
                            dz_list=np.array(d['dz_list']),
                            err_trad=np.array(d['et']), err_mid=np.array(d['em']),
                            order_trad=np.array(d['ot']), order_mid=np.array(d['om']),
                            time_trad=np.array(d['tt']), time_mid=np.array(d['tm']),
                            avg_order_trad=np.float64(d['avg_ot']),
                            avg_order_mid=np.float64(d['avg_om']),
                            label=d.get('label', label))
        print(f"  [EXPORT] {label}_conv.npz"); n_saved += 1

    for key, fname in [('case5','case5.npz'),('case7','case7.npz'),
                        ('case9','case9.npz'),('case11','case11.npz')]:
        if key in case_data:
            np.savez_compressed(os.path.join(RESULTS_DIR, fname), **case_data[key])
            print(f"  [EXPORT] {fname}"); n_saved += 1

    if 'case8' in results_store:
        d8 = results_store['case8']
        np.savez_compressed(os.path.join(RESULTS_DIR, "case8.npz"),
                            ratios=np.array(d8['ratios']),
                            order_trad=np.array(d8['orders_trad']),
                            order_mid=np.array(d8['orders_mid']))
        print("  [EXPORT] case8.npz"); n_saved += 1

    if os.path.exists(os.path.join(RESULTS_DIR, "case12.npz")):
        print("  [EXPORT] case12.npz (saved inline)"); n_saved += 1

    print(f"\n  Total: {n_saved} .npz files in {os.path.abspath(RESULTS_DIR)}/")
    print(sep)


# ==============================================================================
# 11. MAIN
# ==============================================================================
if __name__ == "__main__":
    run_case_0()
    run_case_1(); run_case_2(); run_case_3(); run_case_4()
    run_case_5(); run_case_6(); run_case_7(); run_case_8()
    run_case_9(); run_case_10(); run_case_11(); run_case_12()

    print(sep); print("MERGED PANEL GENERATION"); print(sep_s)

    plot_panel_cases234(
        [results_store['case2'], results_store['case3'], results_store['case4']],
        [r"$P_0=0.60$ W, $N\approx2.0$",
         r"Asymmetric map $60\!:\!40$",
         r"Two-soliton collision, $\Delta\phi=\pi$"],
        "Fig02_Cases234_Convergence_Panel.pdf")

    plot_panel_cases1011(results_store['case10'], results_store['case11'],
                         "Fig10_Cases1011_Generalization_Panel.pdf")

    print(sep); print("GLOBAL EFFICIENCY PARETO  (Case 13)")
    plot_efficiency_pareto(
        [results_store['case1'], results_store['case2'],
         results_store['case3'], results_store['case4']],
        "Fig11_Global_Pareto.pdf")

    export_all_results()

    print("\n" + sep)
    print("  v4.7-Final REFEREE-PROOF VALIDATION COMPLETE  (+ .npz export)")
    print(f"\n  Figures in: {os.path.abspath(FIG_DIR)}/")
    print(f"  Data in:    {os.path.abspath(RESULTS_DIR)}/\n")

    hdr = f"  {'Case':>22} | {'p_Trad':>7} | {'p_Mid':>7} | {'Referee question answered'}"
    print(hdr); print("  " + "-"*(len(hdr)-2))
    for name, key, msg in [
        ("1 Baseline",        'case1',  "N~1.4 breathing -- baseline"),
        ("2 Strong breathing", 'case2',  "Non-integrable dynamics"),
        ("3 Asymmetric 60:40", 'case3',  "Map symmetry independence"),
        ("4 Two-soliton",      'case4',  "Nonlinear interaction"),
        ("6 Efficiency",       'case6',  "Engineering value"),
        ("10 Sech IC",         'case10', "Non-Gaussian initial condition"),
        ("11 Lossy fiber",     'case11', "Loss/gain (alpha!=0)"),
    ]:
        d = results_store[key]
        print(f"  {name:>22} | {d['avg_ot']:7.3f} | {d['avg_om']:7.3f} | {msg}")

    print("\n  STEP-RATIO ROBUSTNESS (Case 8):")
    d8 = results_store['case8']
    for r, ot, om in zip(d8['ratios'], d8['orders_trad'], d8['orders_mid']):
        note = " (float64 noise floor)" if om < 1.0 else ""
        flag_m = "ok" if om > 1.7 else "!"
        print(f"    L_map/dz={r:4d}  p_trad={ot:.3f} ok  p_mid={om:.3f} {flag_m}{note}")

    print("\n  REFEREE ANSWER CHECKLIST:")
    for q, question, answer in [
        ("Q1",  "N=1 only?",           "Cases 2,4 -- N~2 & collision"),
        ("Q2",  "Symmetry needed?",     "Case 3 -- 60:40 asymmetric map"),
        ("Q3",  "Physical mechanism?",  "Case 5 -- local error spike at interface"),
        ("Q4",  "Computational value?", "Case 6 Pareto; Case 9 SS4 comparison"),
        ("Q5",  "Long distance OK?",    "Case 7 -- 50 km, energy+FWHM+peak+momentum"),
        ("Q6",  "dz-ratio dependent?",  "Case 8 -- L_map/dz = 10..640"),
        ("Q7",  "Why not SS4?",         "Case 9 -- equal-time Pareto"),
        ("Q8",  "Gaussian IC only?",    "Case 10 -- sech pulse"),
        ("Q9",  "Lossless only?",       "Case 11 -- 0.2 dB/km lossy fiber"),
        ("Q10", "Proposition 1?",       "Case 12 -- L1_eff sweep (gamma=0), Trad~(1-d), Mid~min(d,1-d), R²≈1"),
    ]:
        print(f"  [{q}] {question:25s} -> {answer}")
    print(sep)

[Font] Serif family: DejaVu Serif
[Font] 'Times New Roman' not found; graceful fallback applied.
  DM SOLITON SSFM -- v4.7-Final REFEREE-PROOF VALIDATION SUITE

[GPU] NVIDIA GeForce RTX 5090 | 31.4 GB total | 30.86 GB free

  gamma=1.317e-03 1/(W*m)  |  L_D=5000 m  |  P0_N1=0.1518 W
  Figures -> /20140171/AI4Science/nonlinearfiber/算法对比/figures_jcp/
--------------------------------------------------------------------------------
CASE 0: REFERENCE SELF-CONSISTENCY CHECK
--------------------------------------------------------------------------------
  Ref1  (dz=0.01562 m) built in 120.0 s
  Ref2  (dz=0.00781 m) built in 239.2 s

  Ref self-error:       3.485e-10
  Trad error (dz=1.0):  4.461e-04   (ratio: 1279869x)
  Mid  error (dz=1.0):  8.457e-08   (ratio: 243x)
  => Ref error is 1279869x smaller than trad -- safe as reference.
CASE 1: BASELINE SYMMETRIC MAP  N≈1.4
--------------------------------------------------------------------------------

[Case 1 (Baseline)]  Ltot=6000 m  L1=50 

Case 1 (Baseline): 100%|██████████████████| 4/4 [00:27<00:00,  6.99s/it]



      dz |   Err Trad |    p_T |    Err Mid |    p_M |  Spdup
  -------|------------|--------|------------|--------|-------
   2.000 |   8.92e-04 |    nan |   3.38e-07 |    nan |   1.01
   1.000 |   4.46e-04 |  1.000 |   8.46e-08 |  2.000 |   0.99
   0.500 |   2.23e-04 |  1.000 |   2.11e-08 |  2.001 |   0.99
   0.250 |   1.12e-04 |  1.000 |   5.27e-09 |  2.004 |   0.98
  => Trad avg order: 1.000  |  Mid avg order: 2.002
  => Error reduction @ dz=1.0: 5275x
[PANEL] -> ./figures_jcp/Fig01_Case1_Baseline_Panel.pdf
CASE 2: STRONG BREATHING  P0=0.60 W  N≈2.0
--------------------------------------------------------------------------------

[Case 2 (Strong)]  Ltot=2000 m  L1=50  L2=50  P0=0.60 W  alpha=0.00e+00
  Ref (dz=0.03125 m) built in 20.5 s


Case 2 (Strong): 100%|████████████████████| 3/3 [00:08<00:00,  2.93s/it]



      dz |   Err Trad |    p_T |    Err Mid |    p_M |  Spdup
  -------|------------|--------|------------|--------|-------
   1.000 |   2.68e-04 |    nan |   3.38e-08 |    nan |   1.03
   0.500 |   1.34e-04 |  1.000 |   8.43e-09 |  2.005 |   1.00
   0.250 |   6.70e-05 |  1.000 |   2.08e-09 |  2.018 |   0.98
  => Trad avg order: 1.000  |  Mid avg order: 2.011
  => Error reduction @ dz=0.5: 15888x
CASE 3: ASYMMETRIC MAP  60:40 DUTY CYCLE
--------------------------------------------------------------------------------

[Case 3 (60:40)]  Ltot=2000 m  L1=60  L2=40  P0=0.30 W  alpha=0.00e+00
  Ref (dz=0.03125 m) built in 20.1 s


...


      dz |   Err Trad |    p_T |    Err Mid |    p_M |  Spdup
  -------|------------|--------|------------|--------|-------
   1.000 |   1.04e-04 |    nan |   9.54e-09 |    nan |   1.01
   0.500 |   5.18e-05 |  1.000 |   2.38e-09 |  2.005 |   1.01
   0.250 |   2.59e-05 |  1.000 |   5.87e-10 |  2.018 |   1.00
  => Trad avg order: 1.000  |  Mid avg order: 2.012
  => Error reduction @ dz=0.5: 21777x
CASE 11: LOSSY FIBER  alpha=0.2 dB/km
--------------------------------------------------------------------------------
  alpha = 0.2 dB/km = 4.6052e-05 Np/m

[Case 11 (Lossy)]  Ltot=2000 m  L1=50  L2=50  P0=0.30 W  alpha=4.61e-05
  Ref (dz=0.03125 m) built in 19.8 s


Case 11 (Lossy): 100%|████████████████████| 3/3 [00:08<00:00,  2.87s/it]



      dz |   Err Trad |    p_T |    Err Mid |    p_M |  Spdup
  -------|------------|--------|------------|--------|-------
   1.000 |   1.18e-04 |    nan |   8.12e-09 |    nan |   1.00
   0.500 |   5.89e-05 |  1.000 |   2.02e-09 |  2.005 |   1.00
   0.250 |   2.95e-05 |  1.000 |   4.99e-10 |  2.019 |   1.00
  => Trad avg order: 1.000  |  Mid avg order: 2.012
  => Error reduction @ dz=0.5: 29137x
CASE 12: INTERFACE SWEEP -- Proposition 1 verification (gamma=0)
--------------------------------------------------------------------------------
  K=10, dz=10.0 m, L_map=100 m, N_maps=20, Ltot=2000 m
  L1_eff = 40 + delta*10  (interface inside step j=4, spans [40,50)m)
  Traditional j=4 eval=40 < L1_eff always → error ∝ (1-delta)
  Midpoint    j=4 eval=45: correct for delta<0.5, wrong for delta>0.5
              → error ∝ min(d,1-d)


Case 12: 100%|██████████████████████████| 49/49 [00:01<00:00, 26.61it/s]



  Traditional: amp=3.4633e-02,  R²=1.000000
  Midpoint:    amp=3.4642e-02,  R²=1.000000
  Trad range:  [6.929e-04, 3.393e-02]
  Mid  range:  [6.929e-04, 1.732e-02]  (min at delta≈0.5: 1.732e-02)
  [EXPORT] -> ./results_v6/case12.npz
[PANEL] -> ./figures_jcp/Fig12_Case12_DeltaSweep_Panel.pdf
MERGED PANEL GENERATION
--------------------------------------------------------------------------------
[PANEL] -> ./figures_jcp/Fig02_Cases234_Convergence_Panel.pdf
[PANEL] -> ./figures_jcp/Fig10_Cases1011_Generalization_Panel.pdf
GLOBAL EFFICIENCY PARETO  (Case 13)
[PLOT] -> ./figures_jcp/Fig11_Global_Pareto.pdf
EXPORTING ALL RESULTS TO .npz
--------------------------------------------------------------------------------
  [EXPORT] case0.npz
  [EXPORT] Case1_conv.npz
  [EXPORT] Case2_conv.npz
  [EXPORT] Case3_conv.npz
  [EXPORT] Case4_conv.npz
  [EXPORT] Case6_conv.npz
  [EXPORT] Case10_conv.npz
  [EXPORT] Case11_conv.npz
  [EXPORT] case5.npz
  [EXPORT] case7.npz
  [EXPORT] case9.npz
  [EXPORT] 

In [1]:
!nvidia-smi

Sun Jun 21 04:02:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5090        On  |   00000000:0C:00.0 Off |                  N/A |
| 55%   30C    P8             15W /  575W |       1MiB /  32607MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pkill python